# Ground Truth Preprocessing for Random Forest

Integrates two spatial datasets to produce labelled tree crown polygons for
Random Forest training and evaluation:

1. **Crown polygons** — output of PyCrown segmentation, filtered in QGIS to
   remove over-segmented crowns.
2. **Municipal tree point data** — point dataset from the City of Enschede
   (`Bomen_gbi.shp`) containing species attributes.

Species are assigned to each crown polygon using a spatial join and a
priority hierarchy:

- **1 point inside crown** — that species is assigned directly.
- **2 points inside crown** — the species of the point closest to the crown
  centroid is used.
- **3 or more points inside crown** — the majority species is used.
- **No allergenic point inside crown** — crown is treated as a private or
  non-target tree and assigned `NaN`.

The output shapefile (`tree_crowns_rf.shp`) is used as input for feature
extraction (`crown_feature_extraction.ipynb`).

**Import libraries**

In [ ]:
import os
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.validation import make_valid

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.paths import INTERIM_DIR, RAW_DIR, PROCESSED_DIR, EXTERNAL_DIR

**Define paths**

In [ ]:
# Crown polygons from PyCrown, filtered in QGIS 
TREE_CROWN_POLYGONS = INTERIM_DIR/ "pycrown_segmentation" / "enschede_pycrown_segmentation_tile34fn2_v01.shp"

# Municipal tree point dataset 
MUNICIPAL_TREE_INVENTORY = EXTERNAL_DIR/ "municipal_gt" / "Bomen_gbi.shp"

**Load and clean tree point data**

In [ ]:
points_gdf = gpd.read_file(MUNICIPAL_TREE_INVENTORY)

# Retain only geometry and species columns
points_gdf = points_gdf[["geometry", "LATBOOMSOO"]]

# Standardise species name to genus only (e.g. "Quercus robur" → "Quercus")
points_gdf["species_name"] = (
    points_gdf["LATBOOMSOO"]
    .str.split(" ")
    .str[0]
    .str.title()
)
points_gdf = points_gdf.drop(columns=["LATBOOMSOO"])

# Keep only the five target allergenic genera
TARGET_SPECIES = ["Betula", "Alnus", "Fraxinus", "Quercus", "Platanus"]
points_gdf = points_gdf[points_gdf["species_name"].isin(TARGET_SPECIES)]

**Load and clean crown polygon data**

In [ ]:
crowns_gdf = gpd.read_file(TREE_CROWN_POLYGONS)

# Keep only geometry and the DN (crown ID) column
crowns_gdf = crowns_gdf[["geometry", "DN"]].rename(columns={"DN": "ID"})
crowns_gdf["ID"] = crowns_gdf["ID"].astype("int")

# Align CRS
if crowns_gdf.crs != points_gdf.crs:
    crowns_gdf = crowns_gdf.to_crs(points_gdf.crs)

# Repair invalid geometries and drop empty ones
crowns_gdf.geometry = crowns_gdf.geometry.apply(make_valid)
crowns_gdf = crowns_gdf[~crowns_gdf.geometry.is_empty]
print(f"Crowns after geometry repair: {len(crowns_gdf)}")

# Remove exact duplicate polygons
crowns_gdf["geom_wkb"] = crowns_gdf.geometry.to_wkb()
crowns_gdf = crowns_gdf.drop_duplicates(subset="geom_wkb").reset_index(drop=True)
crowns_gdf = crowns_gdf.drop(columns=["geom_wkb"])
print(f"Crowns after removing duplicates: {len(crowns_gdf)}")

**Spatial join — match species points to crown polygons**

In [ ]:
# Left join so that crowns without any intersecting point are preserved (NaN species)
crowns_with_points = gpd.sjoin(
    crowns_gdf,
    points_gdf[["species_name", "geometry"]],
    how="left",
    predicate="intersects",
)

# Attach the point geometry for centroid-distance calculations in Case 2
crowns_with_points["point_geom"] = crowns_with_points["index_right"].apply(
    lambda idx: points_gdf.loc[idx, "geometry"] if pd.notna(idx) else None
)

print("Points matched to any crown:", crowns_with_points["index_right"].notna().sum())
print("Unique matched points:", crowns_with_points["index_right"].dropna().nunique())

**Assign species to each crown**

In [ ]:
ALLERGENIC_SET = {"Betula", "Alnus", "Fraxinus", "Quercus", "Platanus"}

# Map species name to integer class ID for Random Forest training
SPECIES_MAP = {
    "Betula":   0,
    "Alnus":    1,
    "Fraxinus": 2,
    "Quercus":  3,
    "Platanus": 4,
}

rows = []

for crown_id, group in crowns_with_points.groupby("ID"):
    group = group.reset_index(drop=True)
    crown_geom = group["geometry"].iloc[0]

    labeled = group[pd.notna(group["species_name"])].copy()

    if len(labeled) == 0:
        # No public tree point inside the crown → private or non-target tree
        species = None

    elif len(labeled) == 1:
        # Case 1: single point — assign its species directly
        species = labeled["species_name"].iloc[0]

    elif len(labeled) == 2:
        # Case 2: two points — use the one closest to the crown centroid
        centroid  = crown_geom.centroid
        distances = gpd.GeoSeries(labeled["point_geom"]).distance(centroid)
        species   = labeled.loc[distances.idxmin(), "species_name"]

    else:
        # Case 3: three or more points — majority vote
        species = labeled["species_name"].mode().iloc[0]

    # is_allergenic: 1 = allergenic species, 0 = non-allergenic public tree,
    #                NaN = no public tree point (private / unlabelled crown)
    is_allergenic = (
        1       if species in ALLERGENIC_SET else
        np.nan  if species is None           else
        0
    )

    rows.append({
        "id":           int(crown_id),
        "species_na":   str(species),
        "is_allerge":   is_allergenic,
        "species_id":   SPECIES_MAP.get(species, np.nan),
        "geometry":     crown_geom,
    })

crowns_trees_gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs=crowns_with_points.crs)

print(f"Total crowns:            {len(crowns_trees_gdf)}")
print(f"  Allergenic species:    {(crowns_trees_gdf['is_allerge'] == 1).sum()}")
print(f"  Non-allergenic public: {(crowns_trees_gdf['is_allerge'] == 0).sum()}")
print(f"  Private / unlabelled:  {crowns_trees_gdf['is_allerge'].isna().sum()}")

**Simplify crown geometries**

In [ ]:
# Simplify polygon edges to remove the blocky staircase effect that arises from
# raster-to-vector conversion. A tolerance of 0.5 m preserves the overall crown
# shape while producing cleaner boundaries.
crowns_trees_gdf["geometry"] = crowns_trees_gdf.geometry.simplify(
    0.5, preserve_topology=True
)

**Save output**

In [ ]:
out_path = os.path.join(PROCESSED_DIR, "tree_shp", "enschede_rf_ground_truth_v01.shp") 
crowns_trees_gdf.to_file(out_path)
print(f"Saved: {out_path}")